# Latihan Individu - Data Transformation

Notebook ini menjawab seluruh poin latihan individu:

1. Penanganan data hilang
2. Encoding menggunakan Label Encoding dan One-Hot Encoding
3. Normalization dan Standardization pada data yang sudah di-encode
4. Visualisasi hasil Normalization dan Standardization disertai analisis
5. Membuat feature baru dari feature yang sudah ada


## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 2. Load Dataset

Pastikan file `mobile price dataset.csv` sudah berada di folder yang sama dengan notebook. Jika di Google Colab, upload file dataset terlebih dahulu.

In [ ]:
# Jika nama file berbeda, sesuaikan pada bagian ini
file_path = 'mobile price dataset.csv'

df = pd.read_csv(file_path)

print('Ukuran dataset:', df.shape)
df.head()

## 3. Informasi Awal Dataset

In [ ]:
print('Informasi dataset:')
df.info()

In [ ]:
print('Jumlah missing value tiap kolom:')
missing_before = df.isnull().sum()
missing_before[missing_before > 0].sort_values(ascending=False)

## 4. Data Cleaning Awal

Beberapa kolom numerik di dataset masih berbentuk teks karena memiliki satuan seperti `mAh`, `mm`, `g`, atau `PPI`. Kolom tersebut dibersihkan supaya bisa diproses sebagai angka.

In [ ]:
df_clean = df.copy()

# Menghapus kolom indeks bawaan jika ada
if 'Unnamed: 0' in df_clean.columns:
    df_clean = df_clean.drop(columns=['Unnamed: 0'])

# Fungsi untuk mengambil angka dari teks
def clean_numeric_column(series):
    return pd.to_numeric(
        series.astype(str).str.replace(r'[^0-9.]', '', regex=True),
        errors='coerce'
    )

# Kolom yang memiliki satuan/teks tetapi seharusnya numerik
numeric_text_cols = [
    'Battery Capacity', 'Width', 'Height', 'Depth', 'Weight',
    'Internal Storage', 'Graphics PPI'
]

for col in numeric_text_cols:
    if col in df_clean.columns:
        df_clean[col] = clean_numeric_column(df_clean[col])

print('Tipe data setelah cleaning awal:')
df_clean[numeric_text_cols].dtypes

## 5. Handling Missing Values

Pada tahap ini, missing value ditangani dengan imputation:

- Kolom numerik diisi menggunakan **mean**.
- Kolom kategorikal diisi menggunakan **mode**.


In [ ]:
df_imputation = df_clean.copy()

# Imputation untuk kolom numerik menggunakan mean
numeric_cols = df_imputation.select_dtypes(include=['int64', 'float64']).columns
for col in numeric_cols:
    df_imputation[col] = df_imputation[col].fillna(df_imputation[col].mean())

# Imputation untuk kolom kategorikal menggunakan mode
categorical_cols = df_imputation.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_imputation[col].isnull().sum() > 0:
        df_imputation[col] = df_imputation[col].fillna(df_imputation[col].mode()[0])

print('Missing value setelah imputation:')
print(df_imputation.isnull().sum().sum())
df_imputation.head()

### Contoh Deletion Menggunakan `dropna()`

Deletion ditampilkan sebagai perbandingan. Pada proses utama, data yang digunakan tetap `df_imputation` agar jumlah data tidak berkurang.

In [ ]:
df_delete = df_clean.copy()
df_delete = df_delete.dropna()

print('Jumlah data awal:', len(df_clean))
print('Jumlah data setelah dropna:', len(df_delete))
print('Jumlah missing value setelah dropna:', df_delete.isnull().sum().sum())

## 6. Encoding

Encoding digunakan untuk mengubah data kategorikal menjadi data numerik.

### 6.1 Label Encoding

Label Encoding digunakan pada beberapa kolom kategorikal agar nilai teks berubah menjadi angka.

In [ ]:
df_label_encoded = df_imputation.copy()

label_encoder = LabelEncoder()
label_encoded_columns = []

# Pilih beberapa kolom kategorikal yang penting dan tidak terlalu panjang untuk contoh Label Encoding
label_cols = [
    'Brand', 'SIM Type', 'Touchscreen', 'OTG Compatible',
    'Quick Charging', 'Operating System', 'Processor Core',
    'Primary Camera Available', 'Secondary Camera Available',
    'Bluetooth Support', 'Wi-Fi', 'GPS Support', 'Smartphone'
]

for col in label_cols:
    if col in df_label_encoded.columns:
        df_label_encoded[col] = label_encoder.fit_transform(df_label_encoded[col].astype(str))
        label_encoded_columns.append(col)

print('Kolom yang dilakukan Label Encoding:')
print(label_encoded_columns)
df_label_encoded[label_encoded_columns].head()

### 6.2 One-Hot Encoding

One-Hot Encoding digunakan untuk mengubah kategori menjadi beberapa kolom bernilai 0 dan 1.

Pada bagian ini, dataset dibatasi pada kolom yang relevan agar hasil encoding tidak terlalu besar.

In [ ]:
# Kolom yang dipakai untuk proses final encoding, scaling, visualisasi, dan feature engineering
selected_cols = [
    'Brand', 'Price', 'Rating', 'No_of_Ratings', 'No_of_Reviews',
    'Display_size_cm', 'Display_size_inches', 'Internal Storage',
    'Battery Capacity', 'Width', 'Height', 'Depth', 'Weight',
    'SIM Type', 'Touchscreen', 'OTG Compatible', 'Quick Charging',
    'Operating System', 'Processor Core', 'Primary Camera Available',
    'Secondary Camera Available', 'Bluetooth Support', 'Wi-Fi',
    'GPS Support', 'Smartphone'
]

selected_cols = [col for col in selected_cols if col in df_imputation.columns]
df_selected = df_imputation[selected_cols].copy()

# One-Hot Encoding untuk semua kolom kategorikal pada data terpilih
df_encoded = pd.get_dummies(df_selected, drop_first=True, dtype=int)

print('Ukuran data sebelum One-Hot Encoding:', df_selected.shape)
print('Ukuran data setelah One-Hot Encoding:', df_encoded.shape)
df_encoded.head()

## 7. Normalization

Normalization menggunakan **MinMaxScaler** untuk mengubah nilai ke rentang 0 sampai 1.

In [ ]:
normal_scaler = MinMaxScaler()

df_normalized = pd.DataFrame(
    normal_scaler.fit_transform(df_encoded),
    columns=df_encoded.columns
)

df_normalized.head()

## 8. Standardization

Standardization menggunakan **StandardScaler** untuk membuat data memiliki rata-rata mendekati 0 dan standar deviasi 1.

In [ ]:
standard_scaler = StandardScaler()

df_standardized = pd.DataFrame(
    standard_scaler.fit_transform(df_encoded),
    columns=df_encoded.columns
)

df_standardized.head()

## 9. Visualisasi Data Setelah Normalization dan Standardization

Visualisasi dilakukan pada beberapa fitur utama agar hasilnya mudah dibaca.

In [ ]:
main_features = [
    'Price', 'Rating', 'No_of_Ratings', 'No_of_Reviews',
    'Battery Capacity', 'Internal Storage', 'Weight'
]
main_features = [col for col in main_features if col in df_encoded.columns]

print('Fitur yang divisualisasikan:', main_features)

In [ ]:
# Visualisasi data setelah Normalization
for col in main_features:
    plt.figure(figsize=(6, 4))
    plt.hist(df_normalized[col], bins=20)
    plt.title(f'Histogram Normalization - {col}')
    plt.xlabel(col)
    plt.ylabel('Frekuensi')
    plt.show()

In [ ]:
# Visualisasi data setelah Standardization
for col in main_features:
    plt.figure(figsize=(6, 4))
    plt.hist(df_standardized[col], bins=20)
    plt.title(f'Histogram Standardization - {col}')
    plt.xlabel(col)
    plt.ylabel('Frekuensi')
    plt.show()

### Analisis Visualisasi

Berdasarkan visualisasi, data hasil **normalization** memiliki rentang nilai 0 sampai 1 sehingga perbedaan skala antar fitur menjadi lebih seimbang. Hal ini berguna karena fitur seperti `Price`, `Battery Capacity`, dan `Weight` memiliki satuan dan rentang nilai yang berbeda.

Pada data hasil **standardization**, nilai fitur diubah sehingga rata-rata mendekati 0 dan standar deviasi menjadi 1. Standardization cocok digunakan ketika model machine learning membutuhkan distribusi data yang lebih stabil dan tidak terlalu dipengaruhi perbedaan skala antar fitur.


## 10. Creating New Feature

Feature baru dibuat dengan memanfaatkan feature yang sudah ada.

In [ ]:
df_feature = df_encoded.copy()

# Menghindari pembagian dengan nol
df_feature['Price_per_Rating'] = df_feature['Price'] / df_feature['Rating'].replace(0, np.nan)
df_feature['Battery_per_Price'] = df_feature['Battery Capacity'] / df_feature['Price'].replace(0, np.nan)
df_feature['Storage_per_Price'] = df_feature['Internal Storage'] / df_feature['Price'].replace(0, np.nan)

# Jika ada nilai NaN dari pembagian, isi dengan 0
df_feature = df_feature.fillna(0)

df_feature[['Price', 'Rating', 'Battery Capacity', 'Internal Storage', 'Price_per_Rating', 'Battery_per_Price', 'Storage_per_Price']].head()

### Analisis Feature Baru

Feature `Price_per_Rating` dapat digunakan untuk melihat perbandingan antara harga dan rating produk. Feature `Battery_per_Price` menunjukkan seberapa besar kapasitas baterai dibandingkan dengan harga. Feature `Storage_per_Price` menunjukkan perbandingan kapasitas penyimpanan internal terhadap harga. Feature baru ini dapat membantu analisis karena memberikan informasi tambahan yang tidak terlihat langsung dari kolom asli.

## 11. Kesimpulan

Data transformation penting dilakukan sebelum data digunakan pada proses machine learning. Pada latihan ini, missing value ditangani menggunakan imputation, data kategorikal diubah menjadi numerik menggunakan Label Encoding dan One-Hot Encoding, kemudian data hasil encoding dilakukan Normalization dan Standardization. Visualisasi menunjukkan bahwa proses scaling membuat skala data menjadi lebih seragam. Selain itu, pembuatan feature baru dapat menambah informasi yang bermanfaat untuk analisis lanjutan.